# Shopping Ads Relevance on Google Colab

This notebook runs the whole project on real data:

1. Install the project from GitHub
2. Quick check on synthetic data
3. Download the Amazon Shopping Queries dataset (ESCI)
4. Run retrieval, the relevance model, ranking and the ads auction
5. (Optional) Let Claude rate query-ad pairs and compare it with the human raters
6. Save the results

Run the cells from top to bottom with **Runtime → Run all**, or one at a time with Shift+Enter. The free CPU runtime is enough, no GPU needed.

## 1. Install the project

In [ ]:
%cd /content
!rm -rf shopping-ads-relevance
!git clone -q https://github.com/Ashish-96-12/shopping-ads-relevance.git
%cd /content/shopping-ads-relevance
!pip install -q .
!shoprel --help

## 2. Quick check on synthetic data

Takes about 15 seconds. If this works, everything is installed correctly.

In [ ]:
!shoprel run --synthetic --n-queries 600 --out reports/synthetic

## 3. Download the ESCI dataset

About 1 GB. Tick `SAVE_TO_DRIVE` to keep a copy in your Google Drive, so next time you can skip the download. Colab will ask for permission to access Drive.

In [ ]:
SAVE_TO_DRIVE = False  # @param {type:"boolean"}

import os

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/esci"
else:
    DATA_DIR = "/content/esci"

os.makedirs(DATA_DIR, exist_ok=True)
!shoprel download --out {DATA_DIR}
!ls -lh {DATA_DIR}

## 4. Run the pipeline on real data

`N_QUERIES` controls the sample size. 5,000 queries runs in a few minutes on the free runtime. Try 20,000 or more for stronger numbers once that works.

In [ ]:
N_QUERIES = 5000  # @param {type:"integer"}
OUT_DIR = f"reports/esci_{N_QUERIES}"

!shoprel run --data-dir {DATA_DIR} --n-queries {N_QUERIES} --out {OUT_DIR}

In [ ]:
from IPython.display import Markdown

Markdown(open(f"{OUT_DIR}/results.md").read())

## 5. (Optional) Claude as a relevance rater

This asks Claude to rate query-ad pairs and checks how often it agrees with the human raters. It uses your Anthropic API key and costs a small amount per pair.

**One-time setup:** click the 🔑 key icon in Colab's left sidebar, add a secret named `ANTHROPIC_API_KEY` with your key from [console.anthropic.com](https://console.anthropic.com), and switch on notebook access. The key stays in your Colab account and never ends up in this notebook.

In [ ]:
RUN_LLM_RATER = False  # @param {type:"boolean"}
N_PAIRS = 100  # @param {type:"integer"}
MODEL = "claude-opus-5-5"  # @param {type:"string"}

if RUN_LLM_RATER:
    from google.colab import userdata

    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    !pip install -q ".[llm]"
    !shoprel llm-rate --data-dir {DATA_DIR} --n-queries {N_QUERIES} --n-pairs {N_PAIRS} --model {MODEL} --out reports/llm_rater
else:
    print("Skipped. Set RUN_LLM_RATER to True to run it.")

## 6. Save the results

Downloads a zip of the `reports` folder. Copy the numbers into the README's "Real ESCI" table, or commit the folder to your repo.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/reports", "zip", "reports")
files.download("/content/reports.zip")